In [5]:
import pandas as pd

# Path to the checkpoint 100 CSV data (update if path is different)
csv_path = "C:/Users/Ratul Sarker/Desktop/CS2_RoundPrediction/data/csv_exports/checkpoint_100.csv"
# Load the checkpoint data into a DataFrame
df = pd.read_csv(csv_path)

# Display the shape and the first few rows to inspect
print(f"Shape of checkpoint 100 data: {df.shape}")
display(df.head())


display(df.columns)



Shape of checkpoint 100 data: (30952, 87)


,ct_alive,t_alive,man_advantage,ct_health_total,t_health_total,ct_health_avg,t_health_avg,health_advantage,ct_armor_total,t_armor_total,...,t_avg_wins,kills_this_round_ct,kills_this_round_t,first_blood_ct,first_blood_t,headshot_kills_ct,headshot_kills_t,damage_dealt_ct,damage_dealt_t,label
0,2.0,2.0,0.0,200.0,200.0,100.0,100.0,0.0,0.0,0.0,...,149.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,2.0,2.0,0.0,200.0,200.0,100.0,100.0,0.0,0.0,200.0,...,149.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,2.0,2.0,0.0,200.0,200.0,100.0,100.0,0.0,0.0,200.0,...,149.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,2.0,2.0,0.0,200.0,200.0,100.0,100.0,0.0,0.0,200.0,...,149.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,2.0,1.0,-1.0,200.0,100.0,100.0,100.0,-100.0,0.0,100.0,...,149.75,1.0,0.0,1.0,0.0,0.0,0.0,108.0,0.0,1


Index(['ct_alive', 't_alive', 'man_advantage', 'ct_health_total',
       't_health_total', 'ct_health_avg', 't_health_avg', 'health_advantage',
       'ct_armor_total', 't_armor_total', 'ct_has_armor_count',
       't_has_armor_count', 'ct_helmet_count', 't_helmet_count',
       'ct_defuser_count', 'ct_rifles', 't_rifles', 'ct_awps', 't_awps',
       'ct_snipers', 't_snipers', 'ct_smgs', 't_smgs', 'ct_shotguns',
       't_shotguns', 'ct_heavy', 't_heavy', 'ct_has_primary', 't_has_primary',
       'rifle_advantage', 'awp_advantage', 'ct_flashbangs', 't_flashbangs',
       'ct_smokes', 't_smokes', 'ct_hegrenades', 't_hegrenades', 'ct_molotovs',
       't_molotovs', 'ct_decoys', 't_decoys', 'ct_utility_total',
       't_utility_total', 'utility_advantage', 'ct_money_total',
       't_money_total', 'ct_money_avg', 't_money_avg', 'ct_equipment_value',
       't_equipment_value', 'equipment_advantage', 'bomb_planted', 'bomb_site',
       'time_since_plant', 'time_elapsed', 'time_remaining', 

# CS2 Round Prediction — Data Exploration

## Data Setup (for anyone reproducing this)

The raw data and processed files are **not included in the repo** (`.gitignore`'d due to size). Follow these steps to set up from scratch:

### Step 1: Download the raw data
Download the **CS2 Demo Dataset** from Kaggle:
- **Dataset:** [CS2 Demo Dataset (CSDS)](https://www.kaggle.com/datasets/bfreskura/csds)
- Download and extract to a local drive (e.g., `D:/CS2_Data/`)
- We use the **December 10, 2023** subset: `D:/CS2_Data/csds/2023/12/10/`

Each match is a folder containing parquet files:
```
D:/CS2_Data/csds/2023/12/10/
├── match_001/
│   ├── player_status      # Health, armor, money per tick
│   ├── player_info        # Team assignments, player IDs
│   ├── player_death       # Kill events
│   ├── player_hurt        # Damage events
│   ├── bomb_state         # Bomb plant/defuse events
│   ├── round_end          # Round outcomes (labels)
│   ├── round_state        # Scores, round timing
│   └── header             # Match metadata (map, etc.)
├── match_002/
│   └── ...
└── ...
```

### Step 2: Update the data path
In `scripts/extract_snapshots.py`, update the `DATA_PATH` variable to point to your downloaded data:
```python
DATA_PATH = Path("D:/CS2_Data/csds/2023/12/10")  # ← change this to your path
```

### Step 3: Run the preprocessing pipeline
```bash
# Extract 86-feature snapshots from raw parquet files
python scripts/extract_snapshots.py
```

This produces:
| File | Description |
|------|-------------|
| `data/X_snapshots.npy` | Feature matrix (N × 86) — one row per mid-round snapshot |
| `data/y_snapshots.npy` | Labels (N,) — 0 = T wins, 1 = CT wins |
| `data/feature_names.json` | List of 86 feature names |

### Step 4: (Optional) Export to CSV for exploration
```python
import numpy as np, pandas as pd, json
X = np.load("data/X_snapshots.npy")
y = np.load("data/y_snapshots.npy")
with open("data/feature_names.json") as f:
    names = json.load(f)
df = pd.DataFrame(X, columns=names)
df["label"] = y
df.to_csv("data/csv_exports/snapshot_data.csv", index=False)
```

### Step 5: Train models
```bash
python scripts/train_mlp.py        # MLP (deep learning)
python scripts/train_xgboost.py    # XGBoost (baseline)
python scripts/tune_mlp.py         # Optuna hyperparameter tuning (optional)
```

---
**Note:** The `models/scaler_params.json` file IS tracked in git — it contains the mean/std values needed for inference without retraining.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Correlation of each feature with label
cs2_correlations = df.drop(columns=["label"]).corrwith(df["label"]).sort_values()

# Display all correlations sorted
print("Feature correlations with label (CT win):\n")
print(cs2_correlations.to_string())
print(f"\nTotal features: {len(cs2_correlations)}")

In [ ]:
# Horizontal bar chart of all feature correlations with label
fig, ax = plt.subplots(figsize=(10, 20))

colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in cs2_correlations.values]
cs2_correlations.plot(kind='barh', ax=ax, color=colors)

ax.set_xlabel('Correlation with CT Win')
ax.set_title('Feature Correlations with Round Outcome')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()